In [1]:
import time

import kagglehub

print("⏳ Downloading PaySim dataset...")
print("   This may take a few minutes depending on your connection.")

start_time = time.time()

path = kagglehub.dataset_download("ealaxi/paysim1")

elapsed = time.time() - start_time

print("✅ Download complete!")
print(f"📁 Dataset path: {path}")
print(f"⏱️ Download time: {elapsed:.1f} seconds")

⏳ Downloading PaySim dataset...
   This may take a few minutes depending on your connection.
✅ Download complete!
📁 Dataset path: /home/dawit-abraham/.cache/kagglehub/datasets/ealaxi/paysim1/versions/2
⏱️ Download time: 9.2 seconds


In [2]:
import os

import numpy as np
import pandas as pd

filename = os.listdir(path)[0]
file_path = os.path.join(path, filename)
df = pd.read_csv(file_path)

In [3]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='str')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [5]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [6]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [7]:
df["isFraud"].value_counts(normalize=True) * 100

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

In [8]:
df.duplicated().sum()

np.int64(0)

In [15]:
df.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,1.781970e+05,5.962370e+05,0.01,13368.395,74684.72,208364.76,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.330,441423.44,1517771.48,10000000.00


Amount sanity check

In [16]:
print("Zero amount:", (df["amount"] == 0).sum())
print("Negative amount:", (df["amount"] < 0).sum())
print("Minimum amount:", df["amount"].min())

Zero amount: 16
Negative amount: 0
Minimum amount: 0.0


In [18]:
zero_amount_stats = (
    df.groupby(df["amount"] == 0)["isFraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

zero_amount_stats["fraud_rate"] *= 100

zero_amount_stats

,transactions,fraud_count,fraud_rate
amount,,,
False,6362604,8197,0.128831
True,16,16,100.000000


outlier × fraud analysis

In [19]:
overall_fraud_rate = df["isFraud"].mean() * 100

print(f"Overall fraud rate: {overall_fraud_rate:.4f}%")

Overall fraud rate: 0.1291%


In [ ]:
numeric_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]



In [23]:
Q1 = df[numeric_features].quantile(0.25)
Q3 = df[numeric_features].quantile(0.75)

IQR = Q3 - Q1

outlier_mask = (
    (df[numeric_features] < (Q1 - 1.5 * IQR)) |
    (df[numeric_features] > (Q3 + 1.5 * IQR))
)

df["is_outlier"] = outlier_mask.any(axis=1)

outlier_fraud = (
    df.groupby("is_outlier")["isFraud"]
      .agg(
          transactions="count",
          fraud_count="sum",
          fraud_rate="mean"
      )
)

outlier_fraud["fraud_rate"] *= 100

outlier_fraud

,transactions,fraud_count,fraud_rate
is_outlier,,,
False,4393187,3064,0.069744
True,1969433,5149,0.261446


In [25]:
class_stats = (
    df["isFraud"]
    .value_counts()
    .rename_axis("isFraud")
    .reset_index(name="transactions")
)

class_stats["percentage"] = (
    class_stats["transactions"] / len(df) * 100
)

class_stats

,isFraud,transactions,percentage
0,0,6354407,99.870918
1,1,8213,0.129082


In [26]:
print(f"Fraud rate: {df['isFraud'].mean() * 100:.4f}%")

Fraud rate: 0.1291%


In [27]:
type_stats = (
    df.groupby("type")["isFraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

type_stats["fraud_rate"] *= 100

type_stats.sort_values("fraud_rate", ascending=False)

,transactions,fraud_count,fraud_rate
type,,,
TRANSFER,532909,4097,0.768799
CASH_OUT,2237500,4116,0.183955
CASH_IN,1399284,0,0.000000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000


In [28]:
df[df["isFraud"] == 1]["type"].value_counts()

type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

# Balance behavior

In [29]:
df["orig_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

df["dest_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

In [30]:
balance_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change"
]

balance_stats = (
    df.groupby("isFraud")[balance_features]
    .mean()
    .T
)

balance_stats.columns = ["Legitimate", "Fraud"]

balance_stats

,Legitimate,Fraud
amount,1.781970e+05,1.467967e+06
oldbalanceOrg,8.328287e+05,1.649668e+06
newbalanceOrig,8.559702e+05,1.923926e+05
oldbalanceDest,1.101421e+06,5.442496e+05
newbalanceDest,1.224926e+06,1.279708e+06
orig_balance_change,-2.314152e+04,1.457275e+06
dest_balance_change,1.235048e+05,7.354580e+05


In [31]:
df["orig_balance_error"] = (
    df["oldbalanceOrg"]
    - df["amount"]
    - df["newbalanceOrig"]
)

df["dest_balance_error"] = (
    df["oldbalanceDest"]
    + df["amount"]
    - df["newbalanceDest"]
)

In [32]:
df.groupby("isFraud")[
    ["orig_balance_error", "dest_balance_error"]
].describe()

orig_balance_error                                             \
                     count           mean            std          min   
isFraud                                                                 
0                6354407.0 -201338.558109  606928.890826 -92445516.64   
1                   8213.0  -10692.325265  265146.131130 -10000000.00   

                                                     dest_balance_error  \
               25%       50%       75%           max              count   
isFraud                                                                   
0       -249953.43 -69049.31 -3034.305  1.000000e-02          6354407.0   
1             0.00      0.00     0.000  3.725290e-09             8213.0   

                                                                             \
                  mean           std          min  25%      50%         75%   
isFraud                                                                       
0         54692.231734  4.360026e+05 -75885725.63  0.0  3500.68   29259.805   
1        732509.301069  1.867748e+06  -8875516.29  0.0  2231.46  442722.010   

                      
                 max  
isFraud               
0        13191233.98  
1        10000000.00

In [33]:
flagged_stats = (
    df.groupby("isFlaggedFraud")["isFraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

flagged_stats["fraud_rate"] *= 100

flagged_stats

,transactions,fraud_count,fraud_rate
isFlaggedFraud,,,
0,6362604,8197,0.128831
1,16,16,100.000000


In [34]:
pd.crosstab(
    df["isFlaggedFraud"],
    df["isFraud"],
    margins=True
)

isFraud,0,1,All
isFlaggedFraud,,,
0,6354407,8197,6362604
1,0,16,16
All,6354407,8213,6362620


In [35]:
numeric_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change",
    "orig_balance_error",
    "dest_balance_error"
]

In [36]:
grouped = df.groupby("isFraud")[numeric_features].agg(["mean", "std"])

mean_0 = grouped.loc[0].xs("mean", level=1)
mean_1 = grouped.loc[1].xs("mean", level=1)

std_0 = grouped.loc[0].xs("std", level=1)
std_1 = grouped.loc[1].xs("std", level=1)

smd = (
    (mean_1 - mean_0)
    / np.sqrt((std_1**2 + std_0**2) / 2)
)

feature_effects = pd.DataFrame({
    "Mean_Legitimate": mean_0,
    "Mean_Fraud": mean_1,
    "SMD": smd,
    "Abs_SMD": smd.abs()
}).sort_values("Abs_SMD", ascending=False)

feature_effects

,Mean_Legitimate,Mean_Fraud,SMD,Abs_SMD
orig_balance_change,-2.314152e+04,1.457275e+06,0.872907,0.872907
amount,1.781970e+05,1.467967e+06,0.736355,0.736355
dest_balance_error,5.469223e+04,7.325093e+05,0.499790,0.499790
dest_balance_change,1.235048e+05,7.354580e+05,0.427137,0.427137
orig_balance_error,-2.013386e+05,-1.069233e+04,0.407077,0.407077
newbalanceOrig,8.559702e+05,1.923926e+05,-0.266291,0.266291
oldbalanceOrg,8.328287e+05,1.649668e+06,0.252552,0.252552
oldbalanceDest,1.101421e+06,5.442496e+05,-0.165433,0.165433
newbalanceDest,1.224926e+06,1.279708e+06,0.014442,0.014442


In [37]:
from scipy.stats import ks_2samp

ks_results = []

for feature in numeric_features:
    legit = df.loc[df["isFraud"] == 0, feature]
    fraud = df.loc[df["isFraud"] == 1, feature]

    statistic, p_value = ks_2samp(legit, fraud)

    ks_results.append({
        "Feature": feature,
        "KS_Statistic": statistic,
        "p_value": p_value
    })

ks_results = pd.DataFrame(ks_results)

ks_results.sort_values(
    "KS_Statistic",
    ascending=False
)

,Feature,KS_Statistic,p_value
7,orig_balance_error,0.808392,0.000000e+00
5,orig_balance_change,0.804171,0.000000e+00
1,oldbalanceOrg,0.542525,0.000000e+00
0,amount,0.444136,0.000000e+00
2,newbalanceOrig,0.413745,0.000000e+00
3,oldbalanceDest,0.244726,0.000000e+00
8,dest_balance_error,0.228860,0.000000e+00
6,dest_balance_change,0.191670,2.239402e-264
4,newbalanceDest,0.115234,2.445360e-95


In [38]:
corr_matrix = df[numeric_features + ["isFraud"]].corr()

target_corr = (
    corr_matrix["isFraud"]
    .drop("isFraud")
    .sort_values(key=abs, ascending=False)
)

target_corr

orig_balance_change    0.362472
amount                 0.076688
dest_balance_error     0.055120
dest_balance_change    0.027028
orig_balance_error     0.011283
oldbalanceOrg          0.010154
newbalanceOrig        -0.008148
oldbalanceDest        -0.005885
newbalanceDest         0.000535
Name: isFraud, dtype: float64

In [39]:
feature_corr = df[numeric_features].corr()

feature_corr

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,orig_balance_change,dest_balance_change,orig_balance_error,dest_balance_error
amount,1.000000,-0.002762,-0.007861,0.294137,0.459304,0.102337,0.845964,-0.970660,-0.189928
oldbalanceOrg,-0.002762,1.000000,0.998803,0.066243,0.042029,-0.220297,-0.087032,-0.050502,0.156464
newbalanceOrig,-0.007861,0.998803,1.000000,0.067812,0.041837,-0.267750,-0.094456,-0.056897,0.163161
oldbalanceDest,0.294137,0.066243,0.067812,1.000000,0.976569,-0.047460,0.232316,-0.304256,-0.025460
newbalanceDest,0.459304,0.042029,0.041837,0.976569,1.000000,-0.006451,0.436191,-0.458750,-0.174942
orig_balance_change,0.102337,-0.220297,-0.267750,-0.047460,-0.006451,1.000000,0.169292,0.139860,-0.171737
dest_balance_change,0.845964,-0.087032,-0.094456,0.232316,0.436191,0.169292,1.000000,-0.801148,-0.684207
orig_balance_error,-0.970660,-0.050502,-0.056897,-0.304256,-0.458750,0.139860,-0.801148,1.000000,0.147540
dest_balance_error,-0.189928,0.156464,0.163161,-0.025460,-0.174942,-0.171737,-0.684207,0.147540,1.000000


In [40]:
df[numeric_features].skew().sort_values(ascending=False)

dest_balance_change    32.916341
amount                 30.993949
orig_balance_change    24.630520
oldbalanceDest         19.921758
newbalanceDest         19.352302
oldbalanceOrg           5.249136
newbalanceOrig          5.176884
orig_balance_error    -30.074746
dest_balance_error    -49.202276
dtype: float64

In [41]:
fraud_by_step = (
    df.groupby("step")["isFraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

fraud_by_step["fraud_rate"] *= 100

fraud_by_step.head()

,transactions,fraud_count,fraud_rate
step,,,
1,2708,16,0.590842
2,1014,8,0.788955
3,552,4,0.724638
4,565,10,1.769912
5,665,6,0.902256


In [42]:
fraud_by_step.sort_values(
    "fraud_rate",
    ascending=False
).head(20)

,transactions,fraud_count,fraud_rate
step,,,
727,12,12,100.0
743,8,8,100.0
742,14,14,100.0
410,20,20,100.0
411,16,16,100.0
412,8,8,100.0
414,16,16,100.0
415,16,16,100.0
417,6,6,100.0


In [44]:
df["orig_transaction_count"] = (
    df.groupby("nameOrig")["nameOrig"]
    .transform("count")
)

df["orig_total_amount"] = (
    df.groupby("nameOrig")["amount"]
    .transform("sum")
)

df["orig_avg_amount"] = (
    df.groupby("nameOrig")["amount"]
    .transform("mean")
)

In [45]:
df["dest_transaction_count"] = (
    df.groupby("nameDest")["nameDest"]
    .transform("count")
)

df["dest_total_amount"] = (
    df.groupby("nameDest")["amount"]
    .transform("sum")
)

df["dest_avg_amount"] = (
    df.groupby("nameDest")["amount"]
    .transform("mean")
)

In [46]:
account_features = [
    "orig_transaction_count",
    "orig_total_amount",
    "orig_avg_amount",
    "dest_transaction_count",
    "dest_total_amount",
    "dest_avg_amount"
]

account_behavior = (
    df.groupby("isFraud")[account_features]
    .mean()
    .T
)

account_behavior.columns = [
    "Legitimate",
    "Fraud"
]

account_behavior

,Legitimate,Fraud
orig_transaction_count,1.002932e+00,1.003409e+00
orig_total_amount,1.787340e+05,1.468447e+06
orig_avg_amount,1.781991e+05,1.466394e+06
dest_transaction_count,1.119624e+01,8.095337e+00
dest_total_amount,3.198327e+06,3.556984e+06
dest_avg_amount,1.790205e+05,8.308296e+05


In [47]:
df["hour_of_day"] = df["step"] % 24
type_time = (
    df.groupby(["hour_of_day", "type"])["isFraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
    .reset_index()
)

type_time["fraud_rate"] *= 100

type_time

,hour_of_day,type,transactions,fraud_count,fraud_rate
0,0,CASH_IN,11170,0,0.000000
1,0,CASH_OUT,11997,150,1.250313
2,0,DEBIT,1647,0,0.000000
3,0,PAYMENT,40556,0,0.000000
4,0,TRANSFER,6217,150,2.412739
...,...,...,...,...,...
115,23,CASH_IN,23805,0,0.000000
116,23,CASH_OUT,26168,161,0.615255
117,23,DEBIT,2073,0,0.000000
118,23,PAYMENT,77345,0,0.000000


# SMD

In [48]:
grouped = df.groupby("isFraud")[numeric_features].agg(["mean", "std"])

mean_0 = grouped.loc[0].xs("mean", level=1)
mean_1 = grouped.loc[1].xs("mean", level=1)

std_0 = grouped.loc[0].xs("std", level=1)
std_1 = grouped.loc[1].xs("std", level=1)

smd = (
    (mean_1 - mean_0)
    / np.sqrt((std_1**2 + std_0**2) / 2)
)

feature_separation = pd.DataFrame({
    "Legitimate_Mean": mean_0,
    "Fraud_Mean": mean_1,
    "SMD": smd,
    "Abs_SMD": smd.abs()
}).sort_values("Abs_SMD", ascending=False)

feature_separation

,Legitimate_Mean,Fraud_Mean,SMD,Abs_SMD
orig_balance_change,-2.314152e+04,1.457275e+06,0.872907,0.872907
amount,1.781970e+05,1.467967e+06,0.736355,0.736355
dest_balance_error,5.469223e+04,7.325093e+05,0.499790,0.499790
dest_balance_change,1.235048e+05,7.354580e+05,0.427137,0.427137
orig_balance_error,-2.013386e+05,-1.069233e+04,0.407077,0.407077
newbalanceOrig,8.559702e+05,1.923926e+05,-0.266291,0.266291
oldbalanceOrg,8.328287e+05,1.649668e+06,0.252552,0.252552
oldbalanceDest,1.101421e+06,5.442496e+05,-0.165433,0.165433
newbalanceDest,1.224926e+06,1.279708e+06,0.014442,0.014442


# KS test

In [49]:
from scipy.stats import ks_2samp

ks_results = []

for feature in numeric_features:
    legitimate = df.loc[df["isFraud"] == 0, feature]
    fraud = df.loc[df["isFraud"] == 1, feature]

    statistic, p_value = ks_2samp(legitimate, fraud)

    ks_results.append({
        "Feature": feature,
        "KS_Statistic": statistic,
        "p_value": p_value
    })

ks_results = (
    pd.DataFrame(ks_results)
    .sort_values("KS_Statistic", ascending=False)
)

ks_results

,Feature,KS_Statistic,p_value
7,orig_balance_error,0.808392,0.000000e+00
5,orig_balance_change,0.804171,0.000000e+00
1,oldbalanceOrg,0.542525,0.000000e+00
0,amount,0.444136,0.000000e+00
2,newbalanceOrig,0.413745,0.000000e+00
3,oldbalanceDest,0.244726,0.000000e+00
8,dest_balance_error,0.228860,0.000000e+00
6,dest_balance_change,0.191670,2.239402e-264
4,newbalanceDest,0.115234,2.445360e-95


# Correlation

In [50]:
correlation = (
    df[numeric_features + ["isFraud"]]
    .corr()["isFraud"]
    .drop("isFraud")
    .sort_values(key=abs, ascending=False)
)

correlation

orig_balance_change    0.362472
amount                 0.076688
dest_balance_error     0.055120
dest_balance_change    0.027028
orig_balance_error     0.011283
oldbalanceOrg          0.010154
newbalanceOrig        -0.008148
oldbalanceDest        -0.005885
newbalanceDest         0.000535
Name: isFraud, dtype: float64

In [51]:
df.groupby(["type", "isFraud"])[
    [
        "amount",
        "orig_balance_change",
        "orig_balance_error",
        "dest_balance_change",
        "dest_balance_error"
    ]
].mean()

amount  orig_balance_change  orig_balance_error  \
type     isFraud                                                          
CASH_IN  0        1.689202e+05        -1.689152e+05      -337835.445783   
CASH_OUT 0        1.739172e+05         2.592296e+04      -147994.192774   
         1        1.455103e+06         1.453796e+06        -1306.125231   
DEBIT    0        5.483665e+03         3.485685e+03        -1997.980151   
PAYMENT  0        1.305760e+04         6.378937e+03        -6678.667998   
TRANSFER 0        9.062290e+05         3.317839e+04      -873050.622427   
         1        1.480892e+06         1.460770e+06       -20122.054174   

                  dest_balance_change  dest_balance_error  
type     isFraud                                           
CASH_IN  0              -1.208134e+05        2.897337e+05  
CASH_OUT 0               1.912257e+05       -1.730854e+04  
         1               1.464626e+06       -9.523835e+03  
DEBIT    0               1.986770e+04       -1.438403e+04  
PAYMENT  0               0.000000e+00        1.305760e+04  
TRANSFER 0               9.945851e+05       -8.835610e+04  
         1               2.908027e+03        1.477984e+06